In [ ]:
import sys, os
!{sys.executable} -m pip install -q pylance s3fs pyarrow polars pyfarmhash

sys.path.insert(0, '/home/ubuntu/vastaiFinal')

import polars as pl
import pyarrow as pa
import lance
import farmhash
from preprocess import engineer_features

BUCKET = 'solana-flash-chunks-668889064578-ca-central-1-an'
PREFIX = 'project_chunks'
OUT    = '/home/ubuntu/vastaiFinal/processed.lance'

S3_GLOB = f"s3://{BUCKET}/{PREFIX}/*.feather"
STORAGE = {"region": "ca-central-1"}

print(f'Source : {S3_GLOB}')
print(f'Output : {OUT}')

In [ ]:
# ── PyArrow dataset scan → farmhash filter → collect ─────────────────────────
import pyarrow as pa
import pyarrow.dataset as pad
import pyarrow.feather as feather
import pyarrow.fs as pafs
import pandas as pd
import s3fs

SKIP_CAST = {'block_timestamp', 'tx_date', 'signature', 'wallet'}

s3_pa  = pafs.S3FileSystem(region="ca-central-1")
s3_fs  = s3fs.S3FileSystem()

# build unified schema: all decimal/int/float → float32
first_path = sorted(s3_fs.glob(f"{BUCKET}/{PREFIX}/*.feather"))[0]
with s3_fs.open(first_path, 'rb') as f:
    raw_schema = feather.read_table(f).schema

unified_fields = []
for field in raw_schema:
    if field.name in SKIP_CAST:
        unified_fields.append(field)
    elif pa.types.is_decimal(field.type) or pa.types.is_integer(field.type) or pa.types.is_floating(field.type):
        unified_fields.append(pa.field(field.name, pa.float32()))
    else:
        unified_fields.append(field)
unified_schema = pa.schema(unified_fields)

dataset = pad.dataset(f"{BUCKET}/{PREFIX}", format="ipc",
                      filesystem=s3_pa, schema=unified_schema)

batches = []
total   = 0

for batch in dataset.to_batches(batch_size=200_000):
    df_b = batch.to_pandas()
    mask = df_b['wallet'].map(lambda w: abs(farmhash.fingerprint64(w)) % 30 == 0)
    df_b = df_b[mask]
    if len(df_b):
        batches.append(df_b)
        total += len(df_b)

pdf = pd.concat(batches, ignore_index=True)
del batches

print(f'Rows    : {len(pdf):,}')
print(f'Wallets : {pdf["wallet"].nunique():,}')
print(f'Columns : {list(pdf.columns)}')

In [ ]:
# ── engineer features ─────────────────────────────────────────────────────────
pdf = engineer_features(pdf)

print(f'Shape  : {pdf.shape}')
print(f'Columns: {list(pdf.columns)}')

In [ ]:
# ── write lance ───────────────────────────────────────────────────────────────
table = pa.Table.from_pandas(pdf, preserve_index=False)
lance.write_dataset(table, OUT, mode='overwrite')
del table

ds = lance.dataset(OUT)
print(f'Rows   : {ds.count_rows():,}')
print(f'Schema :\n{ds.schema}')

In [ ]:
# ── NaN counts (run on instance with enough RAM) ──────────────────────────────
import pandas as pd

df_check = ds.to_table().to_pandas()

isna_counts   = df_check.isna().sum()
isnull_counts = df_check.isnull().sum()

null_df = (isna_counts
             .sort_values(ascending=False)
             .to_frame(name='isna_count'))
null_df['isnull_count'] = isnull_counts
null_df['null_pct']     = (null_df['isna_count'] / len(df_check) * 100).round(3)
null_df['match']        = null_df['isna_count'] == null_df['isnull_count']
print(null_df.to_string())